# Busca de hiperparâmetros do pré-processamento no Kraken

Este notebook apenas orquestra os módulos Python. A busca usa exclusivamente WER, mantendo fixos o modelo e a política de avaliação das referências. Todos os candidatos terminam em nlbin nativo e segmentação neural do Kraken. A baseline usa nlbin 0,5 e o mesmo segmentador neural; o controle raw usa os pixels originais. Não há segmentação legada nesta versão. A imagem original também pode vencer a seleção se tiver WER menor. Os controles com imagem original e segmentador neural são apresentados separadamente. Os pesos do modelo não são treinados nem ajustados.

Extraia o projeto no Colab e defina a raiz como diretório de trabalho antes de executar. O dispositivo é selecionado automaticamente conforme a disponibilidade de GPU. As transcrições foram fornecidas, mas não foram integralmente revisadas; os resultados permanecem provisórios. Este notebook pode ser executado independentemente do notebook principal.


In [ ]:
from kraken_pipeline.setup import configure_notebook, ensure_dependencies

configure_notebook()
ensure_dependencies()

## 1. Inicialização do modelo fixo e reserva de páginas

A divisão padrão usa nove páginas para ajuste e três páginas reservadas para avaliação. A divisão é reprodutível, com semente fixa e sem sobreposição. Não consulte os resultados das páginas reservadas enquanto escolhe intervalos, métodos, combinações ou referências.

A avaliação reservada é pequena e as páginas pertencem ao mesmo livro; portanto, não constitui evidência de generalização ampla.


In [ ]:
from kraken_pipeline.setup import default_device
from kraken_pipeline.config import PipelineConfig
from kraken_pipeline.engine import initialize_kraken
from kraken_pipeline.experiments import dataset_images
from kraken_pipeline.model_management import download_multilingual_model
from kraken_pipeline.reporting import show_comparison
from kraken_pipeline.tuning import SearchConfig, COMBINATION_SPACES, BINARIZATION_SPACES, REFINEMENT_SPACES, select_search_result, generate_trials, split_pages, run_search, evaluate_heldout

config = PipelineConfig(recognition_model=download_multilingual_model(variant="medium"), device=default_device())
engine = initialize_kraken(config)
tuning_pages, heldout_pages = split_pages(dataset_images(config), heldout_count=3, seed=42)

## 2. Escolha da busca e conferência do orçamento

A grade padrão contém 17 candidatos: cinco limiares do nlbin (0,2; 0,35; 0,5; 0,65; 0,8), quatro combinações de contraste e limiar, quatro de denoise e limiar e quatro de ampliação e limiar. Cada candidato aplica apenas um tratamento adicional antes da binarização; a etapa opcional seguinte combina métodos.

`grid` avalia toda essa grade, respeitando `max_trials`. `random` amostra sem reposição com semente fixa e usa `n_trials`. Os dois controles acrescentam duas inferências por página. A grade completa nas nove páginas de ajuste exige 171 inferências de página e pode demorar. Para um teste de execução menor, use `method="random"` com `n_trials=2`; isso não percorre todos os limiares.

`SINGLE_SPACES` continua disponível para buscar também CLAHE, iluminação, deskew e dewarp. Parâmetros e algoritmos ficam nos módulos `.py`.


In [ ]:
search = SearchConfig(method="grid", n_trials=12, max_trials=24, seed=42,
                      spaces=BINARIZATION_SPACES + REFINEMENT_SPACES, segmenter="neural")
candidates = generate_trials(config.preprocessing, search)
len(candidates), len(tuning_pages) * (len(candidates) + 2)

## 3. Ajuste dos métodos individuais

O controle neural precisa concluir a execução e ter WER completo. O texto original sem binarização também participa da seleção, evitando impor um pré-processamento que piore o reconhecimento. Candidatos com falhas ou pontuação parcial não podem vencer. Se nenhum candidato reduzir o WER, a baseline fixa permanece selecionada.

Cada candidato começa na imagem original, aplica a preparação na ordem definida e termina com nlbin. Os melhores parâmetros e sua proveniência são exportados para `best.json`.


In [ ]:
search_run = run_search(config, engine, tuning_pages, search)
search_summary = show_comparison(search_run)
search_run / "best.json"

## 4. Busca opcional de combinações

Execute esta etapa somente após analisar os tratamentos individuais. Mantenha as páginas reservadas fora da seleção. Restrinja `COMBINATION_SPACES` no código Python com base nos resultados das páginas de ajuste.

A busca de combinações começa na grade explicitamente definida; ela não herda automaticamente os parâmetros vencedores dos tratamentos individuais. Escolha o `best.json` final usando apenas o WER das páginas de ajuste. Defina `RUN_COMBINATIONS = True` para executar esta etapa.


In [ ]:
RUN_COMBINATIONS = False
if RUN_COMBINATIONS:
    combination_search = SearchConfig(method="random", n_trials=12, max_trials=64, seed=42, spaces=COMBINATION_SPACES, segmenter="neural")
    combination_run = run_search(config, engine, tuning_pages, combination_search)
    combination_summary = show_comparison(combination_run)

## 5. Fixação da configuração e avaliação única nas páginas reservadas

Defina `BEST_FILE` como o `best.json` da busca escolhida. A avaliação rejeita sobreposição com as páginas de ajuste e verifica os pesos e a política de pontuação. O controle neural fixo é avaliado separadamente, evitando que um limiar ajustado do nlbin altere o controle.

Defina `RUN_HELDOUT = True` para executar esta etapa. Não escolha repetidamente os parâmetros com base nos resultados reservados: isso transforma essas páginas em parte do ajuste.


In [ ]:
search_dirs = (search_run, combination_run) if RUN_COMBINATIONS else (search_run,)
BEST_FILE = select_search_result(search_dirs)
RUN_HELDOUT = False
if RUN_HELDOUT:
    heldout_run = evaluate_heldout(config, engine, BEST_FILE, heldout_pages)
    heldout_summary = show_comparison(heldout_run)

## 6. Transferência para o notebook principal

Baixe `BEST_FILE` do Colab, envie-o para o diretório do projeto na sessão do notebook principal e informe seu caminho em `BEST_RESULTS`, no arquivo `kraken_experiments.ipynb`. Os valores dos parâmetros e a ordem das operações são carregados da seleção exportada. Os parâmetros padrão do código não são sobrescritos.

Preserve a pasta completa de resultados para reprodução: `search.json`, `trials.json`, `leaderboard.csv`, `best.json`, controles e configurações/artefatos de cada candidato. O WER permanece provisório até a revisão das referências.

Na validação executada no Colab, 12 candidatos foram avaliados nas nove páginas de ajuste. O candidato selecionado teve WER de 99,9432%, contra 100% da baseline; nas três páginas reservadas, ambos tiveram 100%. Não foi demonstrada melhoria nas páginas reservadas. Uma verificação adicional executou dois candidatos em grade em uma página de ajuste.
